### Initial packages, etc

In [4]:
!wget https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip
!unzip -q maestro-v3.0.0-midi.zip

--2026-11-05 21:49:42--  https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 34.3.0.27, 34.144.171.27, 104.154.125.27, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|34.3.0.27|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 58416533 (56M) [application/octet-stream]
Saving to: ‘maestro-v3.0.0-midi.zip.1’

maestro-v3.0.0-midi 100%[===================>]  55.71M   187MB/s    in 0.3s    

2026-11-05 21:49:43 (187 MB/s) - ‘maestro-v3.0.0-midi.zip.1’ saved [58416533/58416533]

replace maestro-v3.0.0/2004/MIDI-Unprocessed_XP_08_R1_2004_01-02_ORIG_MID--AUDIO_08_R1_2004_01_Track01_wav.midi? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [5]:
!apt-get update -qq
!apt-get install -y -qq fluidsynth fluid-soundfont-gm
!pip install -q pyfluidsynth pretty_midi

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [6]:
import torch
import torch.nn as nn
import pretty_midi
from pathlib import Path
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'

### READING FILES AND TOKENIZE

In [7]:
files = sorted(Path("maestro-v3.0.0").rglob("*.midi"))
print(len(files))   # should be around 1,276
print(files[0])
pm = pretty_midi.PrettyMIDI(str(files[0]))
piano = next(inst for inst in pm.instruments if not inst.is_drum)
type(piano.notes[0].velocity)

1276
maestro-v3.0.0/2004/MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_05_Track05_wav.midi


int

In [8]:
"""#looking at the distribution of velocities
velocities = []
for file in files[123:128]:
    velocity = []
    pm = pretty_midi.PrettyMIDI(str(files[0]))
    piano = next(inst for inst in pm.instruments if not inst.is_drum)
    for note in piano.notes:
        velocity.append(note.velocity)
    velocities.append(velocity)"""

'#looking at the distribution of velocities\nvelocities = []\nfor file in files[123:128]:\n    velocity = []\n    pm = pretty_midi.PrettyMIDI(str(files[0]))\n    piano = next(inst for inst in pm.instruments if not inst.is_drum)\n    for note in piano.notes:\n        velocity.append(note.velocity)\n    velocities.append(velocity)'

In [9]:
"""#CALCULATE MIN AND MAX PITCH AND VELOCITY IF NECESSARY
MAX = 0
MIN = float('inf')
VEL_MAX = 0
VEL_MIN = float('inf')

for file in tqdm(files):
    pm = pretty_midi.PrettyMIDI(str(file))
    piano = pm.instruments[0]
    for note in piano.notes:
        if MAX < note.pitch: MAX = note.pitch
        if MIN > note.pitch: MIN = note.pitch
        if VEL_MAX < note.velocity: VEL_MAX = note.velocity
        if VEL_MIN > note.velocity: VEL_MIN = note.velocity"""

"#CALCULATE MIN AND MAX PITCH AND VELOCITY IF NECESSARY\nMAX = 0\nMIN = float('inf')\nVEL_MAX = 0\nVEL_MIN = float('inf')\n\nfor file in tqdm(files):\n    pm = pretty_midi.PrettyMIDI(str(file))\n    piano = pm.instruments[0]\n    for note in piano.notes:\n        if MAX < note.pitch: MAX = note.pitch\n        if MIN > note.pitch: MIN = note.pitch\n        if VEL_MAX < note.velocity: VEL_MAX = note.velocity\n        if VEL_MIN > note.velocity: VEL_MIN = note.velocity"

CONSTANTS

In [10]:
MAX = 108 #pitch
MIN = 21 #pitch
VEL_MAX = 126 #velocity
VEL_MIN = 1  #velocity

SCALE = 0.001 #time
MAXTIME = 100 #for time blocks
VEL_SCALE = 10
pieces = []
VEL_MAX_SCALED = int(round(VEL_MAX / VEL_SCALE))
VEL_MIN_SCALED = int(round(VEL_MIN / VEL_SCALE))

TOKENIZATION

In [11]:
class Note():
    def __init__(self, pitch, time, vel, timing):
        self.pitch = pitch
        self.time = int(round(time / SCALE))
        self.vel = vel
        self.timing = timing

def make_tokens(file):
    j = k = 0
    notes = []
    pm = pretty_midi.PrettyMIDI(str(file))
    piano = next(inst for inst in pm.instruments if not inst.is_drum)
    for note in piano.notes:
            vel_scaled = int(round(note.velocity / VEL_SCALE))
            note_1 = Note(note.pitch, note.start, vel_scaled, 'START')
            note_2 = Note(note.pitch, note.end, vel_scaled, 'END')
            notes.append(note_1)
            notes.append(note_2)
    sorted_notes = sorted(notes, key=lambda x: x.time)

    final_notes = []
    for i in range(len(sorted_notes) - 1):
        final_notes.append(('VEL', sorted_notes[i].vel))
        final_notes.append((sorted_notes[i].timing, sorted_notes[i].pitch))
        time_step = sorted_notes[i+1].time - sorted_notes[i].time
        if time_step != 0:
            while time_step > MAXTIME:
                final_notes.append(('TIME', MAXTIME))
                time_step -= MAXTIME
            final_notes.append(('TIME', time_step))

    final_notes.append((sorted_notes[-1].timing, sorted_notes[-1].pitch))
    return final_notes

IDS

In [12]:
def make_vocab():
    vocab = {'PAD': 0, 'BOS': 1, 'EOS': 2}

    for kind in ('START', 'END'):
        for p in range(MIN, MAX+1):
            vocab[(kind, p)] = len(vocab)

    for t in range(1, MAXTIME+1):
        vocab[('TIME', t)] = len(vocab)

    for v in range(VEL_MIN_SCALED, VEL_MAX_SCALED+1):
        vocab[('VEL', v)] = len(vocab)

    return vocab

def make_ids(tokens, vocab):
    ids = [vocab['BOS']]
    for key, val in tokens:
        ids.append(vocab[(key, val)])
    ids.append(vocab['EOS'])
    return ids

vocab = make_vocab()
vocab_size = len(vocab)

In [13]:
#CREATE TOKEN SQS
"""token_sqs = [make_tokens(file) for file in tqdm(files)]
id_sqs = [np.array(make_ids(sq, vocab), dtype=np.uint16) for sq in token_sqs]
lengths = np.array([len(a) for a in id_sqs])

from google.colab import drive
drive.mount('/content/drive')
np.savez('/content/drive/MyDrive/maestro_tokens.npz',
         tokens=np.concatenate(id_sqs),
         lengths=lengths)

#GET INFO OF TOKEN SQS
token_sqs_lens = [len(token_sq) for token_sq in token_sqs]

MAX_PIECE = max(token_sqs_lens)
MIN_PIECE = min(token_sqs_lens)
MEAN_PIECE = sum(token_sqs_lens)/len(token_sqs_lens)

print(f"MAX: {MAX_PIECE} MIN: {MIN_PIECE} MEAN: {MEAN_PIECE} TOTAL TOKENS {sum(token_sqs_lens)}")"""

'token_sqs = [make_tokens(file) for file in tqdm(files)]\nid_sqs = [np.array(make_ids(sq, vocab), dtype=np.uint16) for sq in token_sqs]\nlengths = np.array([len(a) for a in id_sqs])\n\nfrom google.colab import drive\ndrive.mount(\'/content/drive\')\nnp.savez(\'/content/drive/MyDrive/maestro_tokens.npz\',\n         tokens=np.concatenate(id_sqs),\n         lengths=lengths)\n\n#GET INFO OF TOKEN SQS\ntoken_sqs_lens = [len(token_sq) for token_sq in token_sqs]\n\nMAX_PIECE = max(token_sqs_lens)\nMIN_PIECE = min(token_sqs_lens)\nMEAN_PIECE = sum(token_sqs_lens)/len(token_sqs_lens)\n\nprint(f"MAX: {MAX_PIECE} MIN: {MIN_PIECE} MEAN: {MEAN_PIECE} TOTAL TOKENS {sum(token_sqs_lens)}")'

### CREATING THE DATA TENSOR



In [14]:
from google.colab import drive
drive.mount('/content/drive')

d = np.load('/content/drive/MyDrive/maestro_tokens.npz')
tokens = d['tokens']          # read once
lengths = d['lengths']

offsets = np.concatenate([[0], np.cumsum(lengths)])
id_sqs = [tokens[offsets[i]:offsets[i+1]] for i in range(len(lengths))]

Mounted at /content/drive


In [15]:
from torch.utils.data import Dataset, DataLoader
import random

#CREATING THE TENSOR
class MidiDataset(Dataset):
    def __init__(self, ids, sq_len, dataset_len, vocab):
        self.ids = [torch.from_numpy(sq).to(torch.long) for sq in ids]
        self.len = dataset_len
        self.sq_len = sq_len

    def __len__(self):
        return self.len

    def __getitem__(self, i):
        sq = random.choice(self.ids)
        idx = random.randint(0, max(0, len(sq) - self.sq_len))
        sq = sq[idx:idx+self.sq_len]
        sq = torch.nn.functional.pad(sq, (0, self.sq_len - len(sq)), value=0)
        return sq

def make_xy(samples):
    samples = torch.stack(samples)
    return samples[..., :-1], samples[..., 1:]

In [16]:
n = 0.9
sq_len = 1024*2
dataset_len_train = 1024*2
dataset_len_test = 500

id_sqs_train = id_sqs[:int(len(id_sqs)*n)]
id_sqs_test = id_sqs[int(len(id_sqs)*n):]

data_train = MidiDataset(
    ids=id_sqs_train,
    sq_len=sq_len,
    dataset_len=dataset_len_train,
    vocab=vocab
    )

data_test = MidiDataset(
    ids=id_sqs_test,
    sq_len=sq_len,
    dataset_len=dataset_len_test,
    vocab=vocab
    )

data_train_loader = DataLoader(data_train, batch_size=16, shuffle=True, collate_fn=make_xy)
data_test_loader = DataLoader(data_test, batch_size=16, shuffle=False, collate_fn=make_xy)

### TRAIN TEST FUNC

In [25]:
from torch.optim.lr_scheduler import StepLR

def train_and_test(model, loss_fn, optimizer, epochs, dataloader_train, dataloader_test):
    loss_list = []
    loss_test_list = []
    scheduler = StepLR(optimizer, step_size=10, gamma=0.3)
    for epoch in tqdm(range(epochs)):
        loss_epoch = training(model, loss_fn, optimizer, epochs, dataloader_train)
        loss_test_epoch = testing(model, loss_fn, epochs, dataloader_test)
        scheduler.step()

        loss_list.append(loss_epoch)
        loss_test_list.append(loss_test_epoch)
        print(f"\nTRAIN LOSS {loss_epoch} TEST LOSS {loss_test_epoch}")
    return (loss_list, loss_test_list)

def training(model, loss_fn, optimizer, epochs, dataloader):
    model.train()
    loss_epoch = 0
    for xb, yb in dataloader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        B, N = yb.shape
        yb = yb.reshape(B*N)
        y_pred = model(xb)
        loss = loss_fn(y_pred, yb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        loss_epoch += loss.item()
    return loss_epoch/len(dataloader)

@torch.no_grad()
def testing(model, loss_fn, epochs, dataloader):
    model.eval()
    loss_epoch = 0
    for xb, yb in dataloader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        B, N = yb.shape
        yb = yb.reshape(B*N)
        y_pred = model(xb)
        loss = loss_fn(y_pred, yb)
        loss_epoch += loss.item()
    return loss_epoch/len(dataloader)

import matplotlib.pyplot as plt

def plot_loss(loss):
    plt.plot(loss[0], label="train")
    plt.plot(loss[1], label="test")
    plt.legend()

### AUDIO GENERATION

In [26]:
from IPython.display import Audio, display

@torch.no_grad()
def generate_bigrammodel(bigrammodel, max_tokens, idx=1): #idx=1=BOS
    bigrammodel = bigrammodel.to('cpu')
    sq = [idx]
    for _ in range(max_tokens):
        idx = torch.tensor(idx, dtype=torch.long).unsqueeze(dim=-1).unsqueeze(dim=-1)
        logits = bigrammodel(idx)
        logits = logits
        probs = nn.functional.softmax(logits, dim=-1)
        idx = torch.multinomial(probs, num_samples = 1).item()
        sq.append(idx)
    return sq

@torch.no_grad()
def generate_selfattention(model, max_tokens, idx=1):
    sq_res = [idx]
    model.eval()
    cache = [{} for _ in range(model.n_layer)]
    for _ in range(max_tokens):
        idx = torch.tensor(sq_res[-1], dtype=torch.long).unsqueeze(dim=0).unsqueeze(dim=0).to(device)
        logits = model(idx, cache)
        logits = logits[-1]
        probs = nn.functional.softmax(logits, dim=-1)
        idx = torch.multinomial(probs, num_samples = 1).item()
        sq_res.append(idx)
    return sq_res

@torch.no_grad()
def generate_selfattention_sq(model, max_tokens, sq=None):
    sq_res = list(sq[-sq_len:]) if sq is not None else [1]
    model.eval()
    cache = [{} for _ in range(model.n_layer)]
    idx = torch.tensor(sq_res, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_tokens):
        logits = model(idx, cache)
        logits = logits[-1]
        probs = nn.functional.softmax(logits, dim=-1)
        idx = torch.multinomial(probs, num_samples = 1).item()
        sq_res.append(idx)
        idx = torch.tensor(sq_res[-1], dtype=torch.long).unsqueeze(dim=0).unsqueeze(dim=0).to(device)
    return sq_res


def ids_to_tokens(ids, vocab):
    inv = {v: k for k, v in vocab.items()}
    tokens = []
    for i in ids:
        tok = inv[int(i)]
        if tok in ('PAD', 'BOS'):
            continue
        if tok == 'EOS':
            break
        tokens.append(tok)
    return tokens

from collections import deque
def tokens_to_midi(tokens, path='out.mid'):
    pm = pretty_midi.PrettyMIDI()
    inst = pretty_midi.Instrument(program=0)
    t = 0                                # clock, in quantized ticks
    active = {}                          # pitch -> start tick
    vels = deque()                       #active[pitch] = time vels[pitch] = velocity

    def close(pitch, end_tick):
        start = active.pop(pitch)
        vel = vels.popleft()
        if end_tick > start:             # skip zero-length notes
            inst.notes.append(pretty_midi.Note(
                velocity=vel*VEL_SCALE+1, #1-9 velocites are 1 no 0
                pitch=pitch,
                start=start*SCALE,
                end=end_tick*SCALE))

    for kind, val in tokens:
        if kind == 'TIME': #val is time
            t += val

        elif kind == 'START': #val is pitch
            if val in active:            # same pitch already sounding
                continue
            active[val] = t

        elif kind == 'END': #val is pitch
            if val in active:            # ignore ENDs with no matching START
                close(val, t)

        elif kind == 'VEL':
            vels.append(val)

    for pitch in list(active):           # close notes left hanging at the end
        close(pitch, t)

    pm.instruments.append(inst)
    pm.write(path)
    return pm

def make_audio(pm):
    sf2 = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
    audio = pm.fluidsynth(fs=44100, synthesizer=sf2)
    display(Audio(audio, rate=44100))
    return audio

def ids_to_audio(ids, vocab=vocab, path='out.mid'):
    tokens = ids_to_tokens(ids, vocab)
    pm = tokens_to_midi(tokens, path)
    return make_audio(pm)

### SLEF ATTENTION

In [33]:
import torch.nn as nn
torch.manual_seed(1502)
n_embed = 512
dropout=0.15
window_size = sq_len

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, G=4):
        assert num_heads % G == 0
        assert head_size % 4 == 0
        assert num_heads * head_size == n_embed
        #G: groups headsize * G = n_embed // K
        ckv_size = head_size
        super().__init__()
        assert num_heads % G == 0
        self.groups = G
        self.num_heads = num_heads
        self.head_size = head_size
        self.ckv = nn.Linear(n_embed, ckv_size, bias=False)
        self.vup = nn.Linear(ckv_size, head_size*G, bias=False)
        self.kup = nn.Linear(ckv_size, head_size*G//2, bias=False)
        self.key = nn.Linear(n_embed, head_size//2, bias=False)
        self.query = nn.Linear(n_embed, n_embed//2, bias=False)
        self.query_rope = nn.Linear(n_embed, n_embed//2, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(sq_len, sq_len)))
        self.register_buffer('i', torch.arange(self.head_size//4))

        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, cache=None):
        B,T,N = x.shape #B, T, N
        ckv = self.ckv(x) # (B, T, ckv_size)
        k_r = self.key(x)   # (B,T,H/2)
        q = self.query(x) # (B,T,N)
        q_r = self.query_rope(x)
        k_r = k_r.reshape(B, T, 1, self.head_size//2).transpose(1,2) #(B, 1, T, H/2)
        q = q.reshape(B, T, self.num_heads, self.head_size//2).transpose(1,2) #(B, NH, T, H/2)
        q_r = q_r.reshape(B, T, self.num_heads, self.head_size//2).transpose(1,2) #(B, NH, T, H/2)

        past = cache.get('past', 0) if cache is not None else 0
        pos = torch.arange(past, past + T, device=x.device) #(T,)
        if cache is not None:
            cache['past'] = past + T

        angle = torch.outer(pos, 10000**(-4*self.i/self.head_size)) #(T, H/4)
        c = torch.cos(angle)
        s = torch.sin(angle)
        k1, k2 = k_r[..., 0::2], k_r[...,1::2]
        k_r = torch.stack([k1*c - k2*s, k1*s + k2*c], dim=-1).flatten(-2) #(B, NH, T, H/2)
        q1, q2 = q_r[..., 0::2], q_r[...,1::2]
        q_r = torch.stack([q1*c - q2*s, q1*s + q2*c], dim=-1).flatten(-2) #(B, NH, T, H/2)
        q = torch.cat([q, q_r], dim=-1)

        #KV CACHE
        if cache is not None:
            if 'k' in cache:
                k_r = torch.cat([cache['k'], k_r], dim=2)[:, :, -window_size:]
                ckv = torch.cat([cache['ckv'], ckv], dim=1)[:, -window_size:]
            cache['k'], cache['ckv'] = k_r, ckv #k: (B, 1, S, H)

        k = self.kup(ckv)
        v = self.vup(ckv)
        k = k.reshape(B, T, self.groups, self.head_size//2).transpose(1,2) #(B, G, S, H/2)
        v = v.reshape(B, T, self.groups, self.head_size).transpose(1,2) #(B, G, S, H)

        n = self.num_heads // self.groups
        k = torch.concat([k, k_r.repeat_interleave(self.groups, dim=1)], dim=-1)   #(B, G, S, H)
        k = k.repeat_interleave(n, dim=1) #(B, NH, S, H)
        v = v.repeat_interleave(n, dim=1)

        S = k.shape[-2]
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5 # (B, NH, T, S)
        wei = wei.masked_fill(self.tril[S-T:S, :S] == 0, float('-inf'))
        wei = wei.softmax(dim=-1)
        x = wei @ v #(B, NH, T, S) @ (B, 1, S, H) = (B, NH, T, H)

        x = x.transpose(1, 2).reshape(B, T, n_embed)
        x = self.proj(x)
        x = self.dropout(x)
        return x

class FeedFoward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, NHead):
        super().__init__()
        head_size = n_embd // NHead
        self.sa = MultiHeadAttention(NHead, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x, cache=None):
        x = x + self.sa(self.ln1(x), cache)
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramModel2(nn.Module):
    def __init__(self, vocab_size, n_layer=8, NHead=8):
        super().__init__()
        self.n_layer = n_layer
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.blocks = nn.ModuleList([Block(n_embed, NHead=NHead) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embed) # final layer norm
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, cache=None):
        x = self.token_embedding_table(idx) #B, T, N_EMBED
        for i in range(self.n_layer):
            x = self.blocks[i](x, cache[i] if cache is not None else None)
        x = self.ln_f(x) # (B,T,N)
        logits = self.lm_head(x) # (B,T,vocab_size)

        B, T, L = logits.shape #BATCH, TOKEN, LOGIT
        logits = logits.reshape(B*T, L)
        return logits

In [34]:
m = BigramModel2(vocab_size).to(device)

### Training

In [35]:
loss_fn = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(params=m.parameters(), lr=1e-3)
epochs = 30
loss = train_and_test(
    model=m,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=epochs,
    dataloader_train=data_train_loader,
    dataloader_test=data_test_loader)

  3%|▎         | 1/30 [00:58<28:08, 58.23s/it]


TRAIN LOSS 3.61561375670135 TEST LOSS 3.3485900834202766


  7%|▋         | 2/30 [01:56<27:08, 58.16s/it]


TRAIN LOSS 3.2516896426677704 TEST LOSS 2.9576370045542717


 10%|█         | 3/30 [02:54<26:09, 58.14s/it]


TRAIN LOSS 2.7700387686491013 TEST LOSS 2.654627963900566


... (42 outputs left out)

 83%|████████▎ | 25/30 [24:12<04:50, 58.10s/it]


TRAIN LOSS 2.1070285756140947 TEST LOSS 2.1387247890233994


 83%|████████▎ | 25/30 [24:27<04:53, 58.69s/it]


KeyboardInterrupt: 

In [36]:
from google.colab import drive
drive.mount('/content/drive')

import os, torch
os.makedirs('/content/drive/MyDrive/midi_project', exist_ok=True)
path = '/content/drive/MyDrive/midi_project/model.pt'

torch.save(m.state_dict(), path)
print(os.path.getsize(path) / 1e6, "MB")   # verify it was written

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
221.876447 MB


In [ ]:
plot_loss(loss)

In [ ]:
from scipy.io import wavfile
from google.colab import files

for i in range(5):
    ids = generate_selfattention(model=m, max_tokens=2000)
    audio = ids_to_audio(ids)
    wavfile.write(f'out{i}.wav', 44100, audio.astype(np.float32))
    files.download(f'out{i}.wav')

In [ ]:
pm = pretty_midi.PrettyMIDI(str(files[0]))
tokens = make_tokens(files[0])[:500]
ids = make_ids(tokens, vocab)
ids = generate_selfattention_sq(model=m, max_tokens=2000, sq=ids[:-1])
audio = ids_to_audio(ids)

In [ ]:
ids

In [ ]:
import os
print(os.path.exists("/usr/share/sounds/sf2/FluidR3_GM.sf2"))

In [ ]:
pm = pretty_midi.PrettyMIDI(str(files[0]))
audio = make_audio(pm)
import numpy as np
audio = pm.fluidsynth(fs=44100, sf2_path="/usr/share/sounds/sf2/FluidR3_GM.sf2")





In [ ]:
sf2 = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
audio = pm.fluidsynth(fs=44100, sf2_path=sf2)
print(audio.shape, audio.dtype, abs(audio).max())
audio = pm.fluidsynth(fs=44100, sf2_path=sf2)
print(audio.shape, audio.dtype, abs(audio).max())

To fix: